# Validate the Integration and Registry Pipeline

Load and submit the existing six-step pipeline: merge, transform, train, predict,
compare, and registry registration. The cluster, its managed-identity Client ID,
and the external registry come directly from this suite's `.env`; only the
default datastore is discovered from the workspace. No identity lookup is needed.

The final `register_job` uploads the mounted MLflow model directly to the
registry using the **compute cluster UAMI**, not the notebook caller or workspace
identity. It requires **AzureML Registry User** for that identity and private
connectivity to the registry and its backing Storage/ACR. Network Connection
Approver alone does not grant asset permissions. No role is granted here.

Executing the submission cell always starts the full pipeline, including
non-cacheable registry registration. The registry assigns an immutable version,
the step verifies it, and `register_metadata/registration.json` records its ID
and version. Native sharing of existing workspace assets is a separate flow
in foundation 08.


In [ ]:
from pathlib import Path
import json
import pandas as pd
from azure.ai.ml import MLClient, load_job
from azure.ai.ml.entities import ManagedIdentityConfiguration
from azure.identity import AzureCliCredential, ManagedIdentityCredential
from dotenv import dotenv_values

notebook_path = Path(globals().get("__vsc_ipynb_file__", Path.cwd() / "02_submit_integration_compare.ipynb"))
SUITE_ROOT = notebook_path.resolve().parents[1]
env = dotenv_values(SUITE_ROOT / ".env", encoding="utf-8-sig", interpolate=False)

SUBSCRIPTION_ID = (env.get("AZURE_SUBSCRIPTION_ID") or "").strip()
RESOURCE_GROUP = (env.get("AZURE_RESOURCE_GROUP") or "").strip()
WORKSPACE_NAME = (env.get("AZUREML_WORKSPACE_NAME") or "").strip()
REGISTRY_NAME = (env.get("AZUREML_REGISTRY_NAME") or "").strip()
COMPUTE_NAME = (env.get("AZUREML_COMPUTE_NAME") or "").strip()
COMPUTE_IDENTITY_CLIENT_ID = (env.get("AZUREML_COMPUTE_IDENTITY_CLIENT_ID") or "").strip()
if not all((SUBSCRIPTION_ID, RESOURCE_GROUP, WORKSPACE_NAME, REGISTRY_NAME, COMPUTE_NAME, COMPUTE_IDENTITY_CLIENT_ID)):
    raise ValueError(f"Fill workspace, registry, compute name, and identity Client ID in {SUITE_ROOT / '.env'}.")

use_managed_identity = (env.get("USE_MANAGED_IDENTITY", "true") or "").strip().lower()
if use_managed_identity not in {"true", "false"}:
    raise ValueError("USE_MANAGED_IDENTITY must be true or false in .env.")
credential = ManagedIdentityCredential(process_timeout=60) if use_managed_identity == "true" else AzureCliCredential(process_timeout=60)
ml_client = MLClient(credential, SUBSCRIPTION_ID, RESOURCE_GROUP, WORKSPACE_NAME)
SAMPLE_ROOT = SUITE_ROOT.parents[1] / "workshop"
OUTPUT_DATASTORE = ml_client.datastores.get_default().name
pipeline_path = SAMPLE_ROOT / "pipelines/integration-compare-pipeline.yaml"
print(f"Workspace: {WORKSPACE_NAME}; compute: {COMPUTE_NAME}; registry: {REGISTRY_NAME}")

In [ ]:
job = load_job(
    pipeline_path,
    params_override=[
        {"inputs.automl_compute": COMPUTE_NAME},
        {"inputs.registry": REGISTRY_NAME},
    ],
)
job.settings.default_compute = COMPUTE_NAME
job.settings.default_datastore = OUTPUT_DATASTORE
job.identity = ManagedIdentityConfiguration(client_id=COMPUTE_IDENTITY_CLIENT_ID)
for child_job in job.jobs.values():
    child_job.identity = ManagedIdentityConfiguration(client_id=COMPUTE_IDENTITY_CLIENT_ID)
job.display_name = "Readiness taxi integration compare"
job.tags = {"validation": "workspace-readiness", "operation": "integration-compare"}

print(f"Loaded: {type(job).__name__}")
print(f"Compute: {COMPUTE_NAME}")
print(f"Datastore: {OUTPUT_DATASTORE}")
print(f"Runtime identity: compute cluster UMI {COMPUTE_IDENTITY_CLIENT_ID}")
print(f"External registry: {REGISTRY_NAME}")
print("Prepared only; run the submission cell when ready.")


## Submit Once

This submits all six steps, including registry registration. Rerunning this cell starts another billable pipeline.

In [ ]:
submitted_job = ml_client.jobs.create_or_update(job)
print(f"Submitted: {submitted_job.name}")

## Check Parent and Child Status

This cell only reads the existing run. Replace `submitted_job.name` with a quoted existing run name to reconnect after a kernel restart. This is a snapshot; the next cell waits for completion without log streaming.

In [ ]:
JOB_NAME = submitted_job.name
final_job = ml_client.jobs.get(JOB_NAME)
print(f"Pipeline: {final_job.name}; status: {final_job.status}")
print(f"Studio: {final_job.studio_url}")
child_jobs = list(ml_client.jobs.list(parent_job_name=JOB_NAME))
child_summary = pd.DataFrame(
    [
        {
            "name": child.name,
            "display_name": child.display_name,
            "status": child.status,
        }
        for child in child_jobs
    ]
)
display(child_summary)

## Verify Completed Outputs

Wait up to 20 minutes, checking every 10 seconds, then verify all six children and download the existing registration metadata only after `Completed`. A timeout does not cancel or resubmit the pipeline; rerun this cell to keep waiting.

In [ ]:
import time

deadline = time.monotonic() + 40 * 60
final_job = ml_client.jobs.get(JOB_NAME)
while final_job.status not in {"Completed", "Failed", "Canceled", "NotResponding"}:
    if time.monotonic() >= deadline:
        raise TimeoutError(f"Timed out after 20 minutes waiting for pipeline {JOB_NAME}; last status: {final_job.status}.")
    time.sleep(10)
    final_job = ml_client.jobs.get(JOB_NAME)
if final_job.status in {"Failed", "Canceled", "NotResponding"}:
    raise RuntimeError(f"Pipeline ended with status {final_job.status}; inspect its Studio page.")
if final_job.status == "Completed":
    child_jobs = list(ml_client.jobs.list(parent_job_name=JOB_NAME))
    if len(child_jobs) != 6 or any(child.status != "Completed" for child in child_jobs):
        raise RuntimeError("Not all six child jobs have completed.")
    print(f"Completed outputs: {', '.join(final_job.outputs)}")
    download_dir = SUITE_ROOT / "outputs/integration_compare" / final_job.name
    ml_client.jobs.download(final_job.name, output_name="register_metadata", download_path=download_dir)
    metadata_path = download_dir / "named-outputs/register_metadata/registration.json"
    print(json.loads(metadata_path.read_text(encoding="utf-8")))

## Expected Result

The parent pipeline and all six child jobs complete, producing trained-model, prediction, comparison, and verified external-registry registration metadata outputs.

Next: `../03_h2o_reference/01_create_reference_binary_model.ipynb`.